In [ ]:
from transformers import AutoTokenizer
import json

## Load model tokenizer from Huggingface
With the constraint of ~10GB available memory, Qwen3-4B is a good balance between size and capacity

In [ ]:
MODEL = "Qwen/Qwen3-4B-Instruct-2507"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

## Load dataset
We load the downloaded dataset to determine the formatting of `prompt` and `completion` keys. This will directly affect what processing needs to be done.

*note - "Train" is the split from HuggingFace. We will later split into our own train, val, test.*

In [ ]:
with open("../data/train.jsonl", "r") as f:
    jsonl = json.loads(f.readline())

print(jsonl)

## Split conversion
We need to split the conversation into the user and assistant prompts since they masking strategies are different.

In [ ]:
# Single-turn rows split trivially: messages == [user, assistant].
# This is the trainer-time view TRL masks on; the artifact stays conversational.
def to_prompt_completion(row):
    user, assistant = row["messages"]
    return {"prompt": [user], "completion": [assistant], "tools": row["tools"]}

pc = to_prompt_completion(jsonl)

In [ ]:
# Mirror how TRL derives the masked boundary: render the prompt (with the
# assistant opener) and the full convo, then completion == full minus prompt prefix.
prompt_text = tokenizer.apply_chat_template(
    pc["prompt"], tools=pc["tools"], add_generation_prompt=True, tokenize=False
)
full_text = tokenizer.apply_chat_template(
    pc["prompt"] + pc["completion"], tools=pc["tools"], tokenize=False
)
completion_text = full_text[len(prompt_text):]

print("PROMPT tail (masked -> -100):\n", repr(prompt_text[-80:]))
print("\nCOMPLETION:\n", completion_text)

## Determine maximum prompt length
Due to memory constraints, a small window size must be chosen. We check to see the impact on the amount of data we would lose at different sizes.

In [ ]:
# Token length of each FULL example (prompt + completion), as the trainer sees it.
import numpy as np

def n_tokens(row):
    enc = tokenizer.apply_chat_template(
        row["messages"], tools=row["tools"], tokenize=True
    )
    return len(enc["input_ids"])

with open("../data/train.jsonl") as f:
    lengths = np.array([n_tokens(json.loads(line)) for line in f])

for p in (50, 90, 95, 99):
    print(f"p{p}: {int(np.percentile(lengths, p))}")
print("max:", int(lengths.max()), "| n:", len(lengths))
for cut in (1024, 2048):
    over = lengths > cut
    print(f"> {cut}: {over.sum()} truncated ({over.mean():.1%})")

Top choice: **1024**

**Reasoning:** 1392/46893 (2.96%) are cut out. While it would be best to include them, this tradeoff is acceptable.